# 53장 · SQL SELECT·JOIN·GROUP BY

SQL에서는 WHERE로 행을 고르고, JOIN으로 표를 연결하고, GROUP BY로 그룹별 값을 집계합니다.

가상 도서관 자료에서 질문과 사용자를 ID로 연결해 조회합니다. SQLite로 정렬·JOIN·집계를 재현하고, 준비된 PostgreSQL 환경에서 같은 집계 조건을 확인하는 선택 실습이 이어집니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## WHERE와 ORDER BY로 최근 행 조회

관계형 데이터베이스는 같은 종류의 기록을 표에 저장합니다. course_users에는 사용자 한 명당 한 행, course_turns에는 질문 한 건당 한 행이 있습니다. 작은 예제는 같은 이름의 두 표를 SQLite 메모리에 만들고 SQL의 결과부터 관찰합니다. 강사가 준비한 PostgreSQL의 실제 테이블을 읽는 셀은 마지막에 있습니다.


### SQL 조회 요구 정의

<p>alice가 마지막으로 남긴 질문과 사용자별 질문 수를 조회합니다. 질문 세 건에는 소유자 ID가 있고 표시 이름은 사용자 표에만 있습니다.</p>

<p>첫 셀이 실습에 필요한 표 두 개와 가상 데이터를 모두 만듭니다.</p>


### 최근 질문 한 건을 고르는 세 연산

셀의 과제에서 결과가 만들어지는 논리 순서입니다. 먼저 alice를 고른 뒤 최신 ID를 앞에 두고 한 건만 반환합니다.

1. 전체 질문 · course_turns.id = 1, 2, 3
2. WHERE · owner = 'alice' = 1, 2
3. ORDER BY · id DESC = 2, 1
4. LIMIT · LIMIT 1 = 주말 운영은요?


### 셀 01 · SQLite 테이블과 테스트 데이터

수업용 원자료에서 가져온 질문 세 건을 넣고 alice의 질문만 읽습니다. 이 셀을 다시 실행하면 새 메모리 DB로 시작합니다.


In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE course_users (id TEXT PRIMARY KEY, display_name TEXT NOT NULL);
CREATE TABLE course_turns (id INTEGER PRIMARY KEY, owner TEXT NOT NULL, question TEXT NOT NULL, answer TEXT NOT NULL);
INSERT INTO course_users VALUES ('alice', '학습자 A'), ('bob', '학습자 B');
INSERT INTO course_turns VALUES
  (1, 'alice', '언제 여나요?', '평일 오전 9시입니다.'),
  (2, 'alice', '주말 운영은요?', '제공한 안내에 주말 운영 정보가 없습니다.'),
  (3, 'bob', '이용 절차는요?', '먼저 예약합니다.');
""")

print(db.execute("SELECT question FROM course_turns WHERE owner = 'alice' ORDER BY id").fetchall())

[('언제 여나요?',), ('주말 운영은요?',)]


첫 두 행만 보입니다. WHERE가 bob의 행을 빼고 ORDER BY가 질문 ID의 오름차순으로 돌려줬습니다. SELECT 뒤의 question은 반환할 열입니다.

SQLite 조회 결과의 각 행은 열 한 개짜리 튜플입니다. 화면에서 질문만 표시하려면 행 전체와 그 행의 첫 열을 구분해야 합니다. 또 ‘가장 최근 질문’을 고르는 조건에는 삽입 순서를 추측하는 대신 정렬할 열과 방향을 명시합니다.


#### 조회한 한 행의 마지막 쉼표

<mark>열 하나를 읽어도 결과의 각 행은 튜플일 수 있습니다.</mark> 질문 문자열을 꺼낼 때는 행과 열을 구분하세요. 최신 질문을 고르는 순서는 SQL의 ORDER BY에서 먼저 정하고, 파이썬에서는 그 결과의 열을 읽습니다.

선수 개념 · 2장 원본과 복사본으로 컬렉션 다루기

### 질문 행의 owner와 사용자 행의 id

<p>course_turns의 한 행은 질문 한 번이고 course_users의 한 행은 사용자 한 명입니다. 질문 1번과 2번의 owner는 모두 alice입니다. 이후 두 테이블을 연결하면 질문 행을 복사해 이름을 저장하지 않고도 화면에 학습자 A라고 표시할 수 있습니다.</p>


### SELECT로 question 열만 읽기

<p>SELECT question은 반환할 열을 고릅니다. FROM course_turns는 읽을 테이블을 정합니다. WHERE와 ORDER BY가 어느 행을 어떤 순서로 고를지 정해도 최종 출력에 담기는 것은 question 열입니다.</p>


### WHERE의 사용자 필터

<p>WHERE owner = 'alice'는 세 행 중 소유자가 alice인 두 행을 남깁니다. 한 열만 가져온 셀 01의 SELECT에 answer를 추가하면 아래 표처럼 두 열을 볼 수 있습니다.</p>

<p>owner는 조회 조건에만 사용해도 됩니다. 조건을 빼면 bob의 ‘이용 절차는요?’까지 결과에 들어옵니다.</p>

<table><caption>alice 질문의 예상 결과</caption><thead><tr><th scope="col">question</th><th scope="col">answer</th></tr>
</thead><tbody><tr><td>언제 여나요?</td><td>평일 오전 9시입니다.</td></tr>
<tr><td>주말 운영은요?</td><td>제공한 안내에 주말 운영 정보가 없습니다.</td></tr>
</tbody></table>

<p>AND는 두 조건이 모두 맞는 행을 남깁니다. 조건을 바꿔 owner='nobody'로 실행하면 정상 결과가 0행입니다. 이 결과는 DB에 접속하지 못한 오류와 다릅니다. 0행이 나오면 조건과 데이터가 맞는지 먼저 확인합니다.</p>


### 과제 02 · ORDER BY와 LIMIT

alice의 질문을 최근 ID부터 정렬하고 첫 행만 가져오세요. 질문 문장을 출력합니다.

fetchone()은 결과 한 행을 가져옵니다. 한 열만 골라도 행은 ('주말 운영은요?',)처럼 묶여 있습니다. 뒤에 붙인 [0]이 그 행의 첫 열에서 질문 문자열을 꺼냅니다.


In [2]:
# SELECT 결과를 id 내림차순으로 정렬하고 LIMIT를 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>sql = &quot;SELECT question FROM course_turns WHERE owner = &#x27;alice&#x27; ORDER BY id DESC LIMIT 1&quot;
print(db.execute(sql).fetchone()[0])</code></pre>
<p>실행 결과</p><pre><code>주말 운영은요?</code></pre>
<p>“주말 운영은요?”가 나옵니다. 한 건만 고르는 LIMIT 1 앞에 정렬 방향을 정해야 원하는 질문을 재현할 수 있습니다.</p>
</details>


### ORDER BY와 LIMIT

<p>테이블은 출력 순서를 보장하는 목록이 아닙니다. 특정 순서가 필요하면 ORDER BY를 적습니다. ASC는 작은 값부터, DESC는 큰 값부터이며 ASC를 생략할 수 있습니다. 이 작은 샘플에서 질문 id가 큰 행을 마지막에 저장한 질문으로 다룹니다.</p>

<p>LIMIT 1은 결과를 최대 한 행으로 줄입니다. 정렬 없이 LIMIT만 쓰면 어떤 한 행인지 의도대로 정할 수 없습니다. ‘가장 최근’이라는 요구를 실제 서비스에 적용할 때는 시간 열과 동점 처리도 설계해야 합니다. 오늘 예제에서는 고정된 id 기준을 사용합니다.</p>

<h3>같은 세 행에서 결과를 좁히기</h3>

<p>같은 표에 WHERE, ORDER BY, LIMIT를 차례로 적용한 결과입니다. 각 칸에서 사라진 행과 자리가 바뀐 행을 구별하세요.</p>

<article><small>STEP 01 · course_turns</small><h4>SELECT의 원본 행</h4>
<p>alice의 두 질문과 bob의 한 질문이 있습니다.</p><pre>id1 alice 언제 여나요?
id2 alice 주말 운영은요?
id3 bob 이용 절차는요?</pre></article>

<article><small>STEP 02 · WHERE</small><h4>WHERE의 소유자 조건</h4>
<p>owner 값이 같은 두 행을 남깁니다.</p><pre>id1 alice 언제 여나요?
id2 alice 주말 운영은요?</pre></article>

<article><small>STEP 03 · ORDER BY</small><h4>ID 내림차순 정렬</h4>
<p>같은 두 행의 순서를 바꿉니다.</p><pre>id2 alice 주말 운영은요?
id1 alice 언제 여나요?</pre></article>

<article><small>STEP 04 · LIMIT</small><h4>LIMIT으로 첫 행 선택</h4>
<p>정렬된 결과의 맨 앞 한 행을 반환합니다.</p><pre>id2 alice 주말 운영은요?</pre></article>


## JOIN과 GROUP BY를 이용한 집계

질문 표의 owner와 사용자 표의 id가 같은 행끼리 연결합니다. 두 사용자의 모든 행을 그대로 짝지으면 alice의 질문에 bob 이름이 붙을 수도 있습니다. JOIN의 ON 조건을 먼저 확인하세요.


### 셀 03 · JOIN과 GROUP BY 집계

JOIN으로 이름을 붙인 뒤 사용자별 행 수를 집계합니다.


In [3]:
sql = """SELECT u.display_name, COUNT(*) AS turn_count
FROM course_users AS u
JOIN course_turns AS t ON t.owner = u.id
GROUP BY u.id, u.display_name
ORDER BY u.id"""
print(db.execute(sql).fetchall())

[('학습자 A', 2), ('학습자 B', 1)]


학습자 A는 두 건, 학습자 B는 한 건입니다. 이 INNER JOIN은 질문이 하나도 없는 사용자는 세지 않습니다. 그 사용자까지 표시하려면 LEFT JOIN과 COUNT할 열을 함께 검토하세요.


### JOIN 결과를 사용자별로 묶기

<p>GROUP BY u.id, u.display_name은 같은 사용자의 질문을 한 묶음으로 만듭니다. COUNT(*)는 각 묶음의 행 수를 세어 학습자 A에 2, 학습자 B에 1을 돌려줍니다.</p>


### JOIN의 사용자·질문 연결

<p>질문 행에는 owner 값이 있지만 표시 이름은 사용자 테이블에 있습니다. JOIN은 두 테이블에서 조건에 맞는 행을 짝지어 한 결과로 보여 줍니다. ON t.owner = u.id는 질문의 owner와 사용자의 id가 같은 쌍을 고릅니다.</p>

<p>AS u와 AS t는 테이블 별칭입니다. u.display_name은 사용자 표의 열, t.question은 질문 표의 열이라는 뜻입니다. 두 테이블에 id처럼 같은 이름이 있어도 별칭을 붙이면 어느 값을 뜻하는지 분명해집니다.</p>

<table><caption>이름이 붙은 예상 결과</caption><thead><tr><th scope="col">display_name</th><th scope="col">question</th></tr>
</thead><tbody><tr><td>학습자 A</td><td>언제 여나요?</td></tr>
<tr><td>학습자 A</td><td>주말 운영은요?</td></tr>
<tr><td>학습자 B</td><td>이용 절차는요?</td></tr>
</tbody></table>

<p>사용자 행이 두 개라고 JOIN 결과도 두 행인 것은 아닙니다. alice 한 행이 질문 두 행과 각각 연결되어 두 번 나타납니다. JOIN은 화면에서 같은 이름을 자동으로 하나로 합치지 않습니다. 조건 없이 모든 쌍을 만들면 이 샘플은 사용자 2행 × 질문 3행으로 여섯 쌍이 생깁니다.</p>


## 선택 실습 · PostgreSQL에서 집계 실행

강사가 PostgreSQL의 53장 B1 테이블을 준비하고 psycopg가 있는 커널에 수업용 접속 정보(PGHOST·PGPORT·PGDATABASE·PGUSER·PGPASSWORD)와 HK_COURSE_USE_POSTGRES=1을 설정하면 아래 셀이 같은 SQL을 읽기 전용 트랜잭션에서 실행합니다. 접속 정보는 노트북에 입력할 필요가 없습니다.


### 셀 04 · 선택 실습 · PostgreSQL 집계

교실의 PostgreSQL 테이블을 연결했다면 방금 작성한 집계 쿼리를 그대로 보냅니다. 준비되지 않은 커널에서는 안내만 표시합니다.


In [4]:
from importlib.util import find_spec
import os
pg_ready = find_spec("psycopg") is not None and os.getenv("HK_COURSE_USE_POSTGRES") == "1"
if pg_ready:
    import psycopg
    with psycopg.connect(connect_timeout=3) as connection:
        connection.execute("SET TRANSACTION READ ONLY")
        print(connection.execute(sql).fetchall())
else:
    print("PostgreSQL 연결은 준비된 수업 커널에서 확인하세요.")

PostgreSQL 연결은 준비된 수업 커널에서 확인하세요.


예제 데이터만 있는 PostgreSQL에서는 학습자 A 2건, 학습자 B 1건이 나옵니다. 접속 안내만 출력한 경우 이 셀은 집계를 실행하지 않았습니다. 이미 행을 추가한 DB라면 실제 질문 수와 대조하세요.


### PostgreSQL에서 같은 집계 실행하기

<p>기본 실습은 메모리 SQLite에서 실행합니다. 선택 셀은 psycopg가 설치되고 HK_COURSE_USE_POSTGRES가 1로 설정된 환경에서 PostgreSQL에 접속합니다. 같은 테이블과 행이 준비되어 있다면 두 사용자 집계가 일치하는지 비교할 수 있습니다.</p>


## 복습과 실습 점검

### 과제 05 · 질문이 없는 사용자까지 집계하기

사용자 표에 학습자 C를 추가했다고 가정합니다. 아래 WITH 절은 이번 조회에만 C를 더하므로 다시 실행해도 원래 표는 그대로입니다. C의 질문 수가 0으로 나오도록 조인 방식과 COUNT의 대상을 정하세요.


In [ ]:
users = """WITH listed AS (
    SELECT id, display_name FROM course_users
    UNION ALL SELECT 'clara', '학습자 C'
)"""
# listed와 course_turns를 연결해 이름과 질문 수를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>users = &quot;&quot;&quot;WITH listed AS (
    SELECT id, display_name FROM course_users
    UNION ALL SELECT &#x27;clara&#x27;, &#x27;학습자 C&#x27;
)&quot;&quot;&quot;
counts = db.execute(users + &quot;&quot;&quot;
    SELECT u.display_name, COUNT(t.id)
    FROM listed AS u
    LEFT JOIN course_turns AS t ON t.owner = u.id
    GROUP BY u.id, u.display_name
    ORDER BY u.id
&quot;&quot;&quot;).fetchall()
print(counts)</code></pre>
<p>실행 결과</p><pre><code>[('학습자 A', 2), ('학습자 B', 1), ('학습자 C', 0)]</code></pre>
<p>LEFT JOIN은 C의 행을 남기고 질문 쪽 열을 NULL로 채웁니다. COUNT(t.id)는 그 NULL을 세지 않습니다. COUNT(*)를 쓰면 C에게도 한 건이 잡힙니다.</p>
</details>


## 사용자별 마지막 문의 한 건 조회하기

상담 담당자가 각 사용자의 가장 최근 질문을 한 줄씩 보고 싶어 합니다. 앞에서 만든 대화 테이블을 그대로 사용합니다. 이 연습 데이터에서는 id가 접수 순서대로 증가합니다. 실서비스에서 접수 시각을 따로 기록한다면 그 시각으로 최근 여부를 판단해야 합니다.

### 셀 06 · 가장 큰 접수 번호 찾기

alice에게는 질문이 두 건 있습니다. GROUP BY로 사용자마다 한 행을 만들고 그 안에서 가장 큰 번호를 고르세요.

In [5]:
latest_sql = """
SELECT owner, MAX(id) AS last_id
FROM course_turns
GROUP BY owner
"""
print(db.execute(latest_sql + " ORDER BY owner").fetchall())

[('alice', 2), ('bob', 3)]


alice 옆의 2는 두 번째 문의의 접수 번호입니다. bob 옆의 3도 접수 번호이므로 bob이 질문을 세 번 했다는 뜻으로 읽으면 안 됩니다. MAX로 뽑은 번호를 원래 레코드와 다시 연결해야 질문 문장을 가져올 수 있습니다.

### 과제 07 · 최근 질문 번호를 원문과 연결하기

latest_sql을 괄호 안의 조회문으로 사용하세요. 별칭 recent의 last_id와 course_turns의 id를 연결하고, 사용자·질문을 사용자 순으로 출력하세요.

In [6]:
# recent.last_id와 course_turns의 기본 키를 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>latest_questions = f&quot;&quot;&quot;
SELECT t.owner, t.question
FROM ({latest_sql}) AS recent
JOIN course_turns AS t ON t.id = recent.last_id
ORDER BY t.owner
&quot;&quot;&quot;
print(db.execute(latest_questions).fetchall())</code></pre>
<p>실행 결과</p><pre><code>[(&#x27;alice&#x27;, &#x27;주말 운영은요?&#x27;), (&#x27;bob&#x27;, &#x27;이용 절차는요?&#x27;)]</code></pre>
<p>alice에게는 주말 운영 질문 한 건, bob에게는 이용 절차 질문 한 건이 남습니다. JOIN을 owner만으로 걸면 alice의 첫 질문도 따라옵니다. 두 질문이 보일 때는 정렬보다 먼저 JOIN 조건을 살펴보세요.</p>
</details>

<mark>묶음별 최댓값은 그 행의 나머지 열을 대신 돌려주지 않습니다.</mark> 먼저 사용자별 last_id를 구하고, 기본 키로 원래 행을 찾아야 질문 문장을 정확하게 가져올 수 있습니다.

조건을 만족하는 행을 한 건으로 좁히는 방식은 다음 장에서 문서 수정 대상을 확인할 때 다시 씁니다.